# 🏆 Baseline 2: MobileCLIP (Zero-Shot Multimodal)
## E-commerce Visual Search System — Shopee Dataset (34,250 items)

**Pipeline:**
- 🍎 **Model:** `MobileCLIP` (Apple's lightweight CLIP) — xử lý cả Image lẫn Text trong cùng embedding space
- 🔀 **Fusion:** Linear interpolation: `fused = L2_Norm(α * img_feat + (1-α) * txt_feat)`
- 🔍 **Search:** FAISS `IndexFlatIP` (Cosine Similarity)
- 📊 **Metrics:** mAP@5, Precision@1, Recall@5

**Dataset Split (giống Baseline 1 — STRICT):**
- Gallery: toàn bộ 34,250 ảnh
- Val queries (20%): ~6,850 ảnh → dùng để grid search alpha
- Test queries (80%): ~27,400 ảnh → đánh giá cuối cùng (chạy 1 lần)

> **Fallback:** Nếu MobileCLIP không cài được, notebook tự động dùng `openai/clip-vit-base-patch32` từ HuggingFace làm fallback.

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
# Cài đặt thư viện cơ bản
!pip install -q faiss-gpu Pillow pandas numpy tqdm scikit-learn transformers timm

# Thử cài MobileCLIP từ Apple
import subprocess, sys

USE_MOBILECLIP = False
try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/apple/ml-mobileclip.git'],
        capture_output=True, text=True, timeout=120
    )
    import mobileclip
    USE_MOBILECLIP = True
    print('✅ MobileCLIP (Apple) đã được cài thành công!')
except Exception as e:
    print(f'⚠️  Không cài được MobileCLIP: {e}')
    print('🔄 Sẽ dùng HuggingFace CLIP (openai/clip-vit-base-patch32) làm fallback.')
    !pip install -q transformers

print(f'\n🔧 Chế độ: {"MobileCLIP (Apple)" if USE_MOBILECLIP else "CLIP HuggingFace Fallback"}')

## 📂 Cell 2: Import thư viện & Cấu hình

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
import faiss

# ─── CẤU HÌNH ───────────────────────────────────────────────────────────────
# 🔧 Chỉnh đường dẫn cho phù hợp với môi trường Colab của bạn
DATA_DIR    = '/kaggle/input/shopee-product-matching'  # Thay đổi nếu cần
CSV_PATH    = os.path.join(DATA_DIR, 'train.csv')
IMG_DIR     = os.path.join(DATA_DIR, 'train_images')

# Với MobileCLIP: dùng checkpoint s0 (nhanh nhất)
MOBILECLIP_VARIANT = 'mobileclip_s0'   # s0 / s1 / s2 / b
MOBILECLIP_CKPT    = '/tmp/mobileclip_s0.pt'  # sẽ download nếu cần

BATCH_SIZE  = 64
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

print(f'🔧 Thiết bị đang dùng : {DEVICE}')
print(f'📂 Thư mục dữ liệu  : {DATA_DIR}')
print(f'🍎 MobileCLIP variant: {MOBILECLIP_VARIANT}')

## 📊 Cell 3: Đọc dữ liệu & Chia tập (Strict Split)

In [ ]:
# Đọc file CSV
df = pd.read_csv(CSV_PATH)
print(f'📊 Tổng số mẫu: {len(df)}')
print(f'📋 Các cột: {list(df.columns)}')
print(df.head())

# ─── STRICT DATASET SPLITTING RULE ──────────────────────────────────────────
# Gallery = toàn bộ dataset
df_gallery = df.copy()
print(f'\n🗂️ Gallery size: {len(df_gallery)} ảnh')

# Chia val (20%) và test (80%) — dùng cùng seed với Baseline 1
val_idx, test_idx = train_test_split(
    df.index.tolist(),
    test_size=0.8,
    random_state=RANDOM_SEED,
    stratify=df['label_group']
)

df_val  = df.loc[val_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

print(f'✅ Val queries  (20%): {len(df_val)} ảnh  → grid search alpha')
print(f'✅ Test queries (80%): {len(df_test)} ảnh → đánh giá cuối (1 lần!)')

## 🍎 Cell 4: Tải MobileCLIP (hoặc Fallback CLIP)

In [ ]:
# ─── LOAD MODEL ─────────────────────────────────────────────────────────────

if USE_MOBILECLIP:
    # ── MobileCLIP (Apple) ───────────────────────────────────────────────────
    import mobileclip

    # Download checkpoint nếu chưa có
    if not os.path.exists(MOBILECLIP_CKPT):
        print(f'⏬ Đang download MobileCLIP checkpoint ({MOBILECLIP_VARIANT})...')
        import urllib.request
        ckpt_urls = {
            'mobileclip_s0': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt',
            'mobileclip_s1': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt',
            'mobileclip_s2': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s2.pt',
            'mobileclip_b':  'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_b.pt',
        }
        urllib.request.urlretrieve(ckpt_urls[MOBILECLIP_VARIANT], MOBILECLIP_CKPT)
        print(f'✅ Đã download: {MOBILECLIP_CKPT}')

    print(f'⏳ Đang tải MobileCLIP ({MOBILECLIP_VARIANT})...')
    clip_model, _, preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT,
        pretrained=MOBILECLIP_CKPT
    )
    tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()

    # Lấy embedding dimension
    with torch.no_grad():
        dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
        embed_dim = clip_model.encode_image(dummy).shape[-1]

    MODEL_NAME = f'MobileCLIP ({MOBILECLIP_VARIANT})'
    print(f'✅ {MODEL_NAME} đã sẵn sàng. Embedding dim: {embed_dim}')

else:
    # ── Fallback: HuggingFace CLIP ────────────────────────────────────────────
    from transformers import CLIPProcessor, CLIPModel

    HF_CLIP_NAME = 'openai/clip-vit-base-patch32'
    print(f'⏳ Đang tải HuggingFace CLIP ({HF_CLIP_NAME})...')
    clip_model   = CLIPModel.from_pretrained(HF_CLIP_NAME).to(DEVICE).eval()
    preprocess   = CLIPProcessor.from_pretrained(HF_CLIP_NAME)
    tokenizer    = None  # dùng CLIPProcessor cho cả image lẫn text
    embed_dim    = clip_model.config.projection_dim  # thường 512

    MODEL_NAME = f'CLIP HuggingFace ({HF_CLIP_NAME})'
    print(f'✅ {MODEL_NAME} đã sẵn sàng. Embedding dim: {embed_dim}')

print(f'\n📐 Feature dimension: {embed_dim}')

## 🖼️📝 Cell 5: Hàm trích xuất Image & Text Features

In [ ]:
# ─── Dataset cho ảnh ────────────────────────────────────────────────────────
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, use_mobileclip=True):
        self.df            = df
        self.img_dir       = img_dir
        self.transform     = transform
        self.use_mobileclip = use_mobileclip

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (256, 256), color=(128, 128, 128))

        if self.use_mobileclip and self.transform:
            return self.transform(img)         # Tensor
        else:
            return img                         # PIL Image (HuggingFace xử lý sau)


# ─── Trích xuất Image Features ───────────────────────────────────────────────
def extract_image_features_clip(df_input, img_dir, batch_size=64):
    """Trích xuất image features từ CLIP (MobileCLIP hoặc HuggingFace)."""
    all_feats = []

    if USE_MOBILECLIP:
        dataset = ShopeeImageDataset(df_input, img_dir,
                                     transform=preprocess, use_mobileclip=True)
        loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True)
        with torch.no_grad():
            for imgs in tqdm(loader, desc='🖼️ MobileCLIP image features'):
                imgs = imgs.to(DEVICE)
                feats = clip_model.encode_image(imgs)
                all_feats.append(feats.cpu().float().numpy())
    else:
        # HuggingFace: xử lý từng batch PIL images
        pil_list = []
        for idx in range(len(df_input)):
            row = df_input.iloc[idx]
            img_path = os.path.join(img_dir, row['image'])
            try:
                pil_list.append(Image.open(img_path).convert('RGB'))
            except Exception:
                pil_list.append(Image.new('RGB', (224, 224), (128, 128, 128)))

        for i in tqdm(range(0, len(pil_list), batch_size),
                      desc='🖼️ HuggingFace CLIP image features'):
            batch = pil_list[i:i + batch_size]
            inputs = preprocess(images=batch, return_tensors='pt',
                                padding=True).to(DEVICE)
            with torch.no_grad():
                feats = clip_model.get_image_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())

    return np.vstack(all_feats)  # (N, embed_dim)


# ─── Trích xuất Text Features ────────────────────────────────────────────────
def extract_text_features_clip(df_input, batch_size=256):
    """Trích xuất text features từ CLIP (MobileCLIP hoặc HuggingFace)."""
    titles    = df_input['title'].fillna('').tolist()
    all_feats = []

    if USE_MOBILECLIP:
        for i in tqdm(range(0, len(titles), batch_size),
                      desc='📝 MobileCLIP text features'):
            batch = titles[i:i + batch_size]
            tokens = tokenizer(batch).to(DEVICE)
            with torch.no_grad():
                feats = clip_model.encode_text(tokens)
            all_feats.append(feats.cpu().float().numpy())
    else:
        for i in tqdm(range(0, len(titles), batch_size),
                      desc='📝 HuggingFace CLIP text features'):
            batch  = titles[i:i + batch_size]
            inputs = preprocess(text=batch, return_tensors='pt',
                                padding=True, truncation=True,
                                max_length=77).to(DEVICE)
            with torch.no_grad():
                feats = clip_model.get_text_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())

    return np.vstack(all_feats)  # (N, embed_dim)


print('✅ Hàm trích xuất features đã sẵn sàng')

## 🔀 Cell 6: Fusion & FAISS

In [ ]:
def fuse_and_normalize_clip(img_feats, txt_feats, alpha):
    """
    Linear interpolation fusion (cùng embedding space của CLIP):
        fused = L2_Normalize(alpha * img + (1 - alpha) * txt)
    """
    fused = alpha * img_feats + (1 - alpha) * txt_feats  # (N, embed_dim)
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)
    return (fused / norms).astype(np.float32)


def build_faiss_index(features):
    """Xây dựng FAISS IndexFlatIP cho Cosine Similarity (sau L2-normalize)."""
    dim   = features.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(features)
    return index


print(f'✅ Fusion dùng linear interpolation (cùng embedding space, dim={embed_dim})')

## 📐 Cell 7: Hàm đánh giá Metrics

In [ ]:
def get_ground_truth_dict(df_input):
    """Tạo dict: posting_id → set(posting_id cùng label_group)."""
    gt = {}
    for label, grp in df_input.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids:
            gt[pid] = ids
    return gt


def evaluate_retrieval_clip(query_df, gallery_df, query_img, query_txt,
                             gallery_img, gallery_txt, alpha, K=5):
    """
    Đánh giá retrieval:
        - Fuse + normalize query & gallery
        - Build FAISS IndexFlatIP
        - Top-K+1 nearest neighbors, loại self-match
        - Tính mAP@K, P@1, Recall@K
    """
    q_fused = fuse_and_normalize_clip(query_img, query_txt, alpha)
    g_fused = fuse_and_normalize_clip(gallery_img, gallery_txt, alpha)

    gt_dict = get_ground_truth_dict(gallery_df)

    index = build_faiss_index(g_fused)
    scores, indices = index.search(q_fused, K + 1)

    gallery_pids = gallery_df['posting_id'].tolist()
    ap_list, p1_list, r5_list = [], [], []

    for i, row in enumerate(query_df.itertuples()):
        qid      = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}

        if len(relevant) == 0:
            continue

        # Lấy top-K (bỏ self-match)
        retrieved = []
        for idx in indices[i]:
            pid = gallery_pids[idx]
            if pid != qid:
                retrieved.append(pid)
            if len(retrieved) == K:
                break

        # AP@K
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, start=1):
            if pid in relevant:
                hits += 1
                ap  += hits / rank
        ap /= min(len(relevant), K)
        ap_list.append(ap)

        # P@1
        p1_list.append(1.0 if (retrieved and retrieved[0] in relevant) else 0.0)

        # Recall@K
        r5_list.append(len(set(retrieved) & relevant) / len(relevant))

    return {
        'mAP@5':       np.mean(ap_list),
        'Precision@1': np.mean(p1_list),
        'Recall@5':    np.mean(r5_list),
    }


print('✅ Hàm evaluate_retrieval_clip đã sẵn sàng')

## 🔍 Cell 8: Trích xuất tất cả Features

In [ ]:
# ─── Gallery Features ────────────────────────────────────────────────────────
print('📦 Đang trích xuất GALLERY features...')
gallery_img_feats = extract_image_features_clip(df_gallery, IMG_DIR, BATCH_SIZE)
gallery_txt_feats = extract_text_features_clip(df_gallery)
print(f'✅ Gallery img: {gallery_img_feats.shape} | txt: {gallery_txt_feats.shape}')

# ─── Val Features ────────────────────────────────────────────────────────────
print('\n📦 Đang trích xuất VAL features...')
val_img_feats = extract_image_features_clip(df_val, IMG_DIR, BATCH_SIZE)
val_txt_feats = extract_text_features_clip(df_val)
print(f'✅ Val img: {val_img_feats.shape} | txt: {val_txt_feats.shape}')

# ─── Test Features ────────────────────────────────────────────────────────────
print('\n📦 Đang trích xuất TEST features...')
test_img_feats = extract_image_features_clip(df_test, IMG_DIR, BATCH_SIZE)
test_txt_feats = extract_text_features_clip(df_test)
print(f'✅ Test img: {test_img_feats.shape} | txt: {test_txt_feats.shape}')

## 🎯 Cell 9: Grid Search Alpha trên Validation Set

In [ ]:
# Grid search alpha từ 0.1 → 0.9 trên VAL queries
# KHÔNG được dùng test set ở bước này!

alphas = np.arange(0.1, 1.0, 0.1).round(1)
print(f'🔍 Grid search alpha: {alphas.tolist()}')
print(f'   (alpha=1.0 → chỉ dùng image | alpha=0.0 → chỉ dùng text)')
print('─' * 60)

val_results_2 = []
best_alpha_2  = None
best_map5_2   = -1.0

for alpha in alphas:
    metrics = evaluate_retrieval_clip(
        query_df    = df_val,
        gallery_df  = df_gallery,
        query_img   = val_img_feats,
        query_txt   = val_txt_feats,
        gallery_img = gallery_img_feats,
        gallery_txt = gallery_txt_feats,
        alpha       = alpha,
        K           = 5
    )
    val_results_2.append({'alpha': alpha, **metrics})
    print(f'  alpha={alpha:.1f} | mAP@5={metrics["mAP@5"]:.4f} | '
          f'P@1={metrics["Precision@1"]:.4f} | R@5={metrics["Recall@5"]:.4f}')

    if metrics['mAP@5'] > best_map5_2:
        best_map5_2   = metrics['mAP@5']
        best_alpha_2  = alpha

print('─' * 60)
print(f'\n🏆 BEST_ALPHA_2 = {best_alpha_2:.1f}  (Val mAP@5 = {best_map5_2:.4f})')

# In bảng val kết quả
df_val_results = pd.DataFrame(val_results_2)
print('\n📊 Bảng kết quả Validation:')
print(df_val_results.to_string(index=False))

## 🧪 Cell 10: Đánh giá cuối cùng trên TEST SET

In [ ]:
# ⚠️ CHỈ CHẠY 1 LẦN với BEST_ALPHA_2 tìm được từ val!
print(f'⚠️  Đang đánh giá TEST SET với alpha = {best_alpha_2:.1f}...')
print('⚠️  Lưu ý: bước này chỉ được chạy DUY NHẤT 1 LẦN!')

test_metrics_2 = evaluate_retrieval_clip(
    query_df    = df_test,
    gallery_df  = df_gallery,
    query_img   = test_img_feats,
    query_txt   = test_txt_feats,
    gallery_img = gallery_img_feats,
    gallery_txt = gallery_txt_feats,
    alpha       = best_alpha_2,
    K           = 5
)

print('\n📊 KẾT QUẢ TEST SET — Baseline 2 (MobileCLIP / CLIP Fallback):')
print(f'   mAP@5        = {test_metrics_2["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_2["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_2["Recall@5"]:.4f}')

## 📋 Cell 11: Bảng kết quả Markdown (copy-paste vào báo cáo)

In [ ]:
from IPython.display import Markdown, display

model_name_row = 'MobileCLIP (s0)' if USE_MOBILECLIP else 'CLIP (ViT-B/32)'
feature_dim_2  = str(embed_dim)

table_md = f"""
## 📊 Kết quả so sánh Baseline Models — TEST SET

| Baseline Model | Feature Dim | Best Alpha | Test mAP@5 | Test Precision@1 | Test Recall@5 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| EfficientNetB0 + MiniLM | 1280 + 384 | — | — | — | — |
| {model_name_row} | {feature_dim_2} | {best_alpha_2:.1f} | {test_metrics_2['mAP@5']:.4f} | {test_metrics_2['Precision@1']:.4f} | {test_metrics_2['Recall@5']:.4f} |

> 💡 Điền kết quả Baseline 1 từ notebook `Baseline1_EfficientNetB0_MiniLM.ipynb` vào dòng trên.
"""

display(Markdown(table_md))
print('\n📋 Raw Markdown (copy vào báo cáo):')
print(table_md)

## 📊 Cell 12: Bảng tổng hợp CUỐI CÙNG (chạy sau khi có kết quả cả 2 baseline)

In [ ]:
# ── Điền kết quả Baseline 1 vào đây sau khi chạy notebook kia ───────────────
# (Thay các giá trị bên dưới bằng kết quả thực tế từ Baseline 1)
B1_ALPHA  = 0.0      # thay bằng BEST_ALPHA_1
B1_MAP5   = 0.0000   # thay bằng kết quả thực
B1_P1     = 0.0000
B1_R5     = 0.0000

table_final = f"""
## 📊 KẾT QUẢ CUỐI CÙNG — So sánh 2 Baselines trên TEST SET

| Baseline Model | Feature Dim | Best Alpha | Test mAP@5 | Test Precision@1 | Test Recall@5 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| EfficientNetB0 + MiniLM | 1280 + 384 | {B1_ALPHA:.1f} | {B1_MAP5:.4f} | {B1_P1:.4f} | {B1_R5:.4f} |
| {model_name_row} | {feature_dim_2} | {best_alpha_2:.1f} | {test_metrics_2['mAP@5']:.4f} | {test_metrics_2['Precision@1']:.4f} | {test_metrics_2['Recall@5']:.4f} |
"""

display(Markdown(table_final))
print('\n📋 Raw Markdown (copy vào báo cáo):')
print(table_final)